# Задание 1 (основная часть). Face Alignment — Stacked Hourglass Network

Этот ноутбук закрывает **вторую стадию** стандартного пайплайна распознавания лиц:

> детекция → **выравнивание (alignment)** → распознавание

Выравнивание состоит из двух шагов: (1) найти на лице ключевые точки, (2) геометрически
привести лицо к «каноничному» виду, чтобы глаза/нос/рот у всех лиц оказались примерно в
одних и тех же местах. Это сильно упрощает работу сети-распознавателя на третьей стадии.

**Что делаем в ноутбуке (план):**

1. Готовим рабочий датасет на основе **CelebA In‑the‑Wild** (10 000+ изображений), обосновываем критерии отбора, кропаем лица по bbox, сохраняем CSV с исходными именами.
2. Превращаем 5 ключевых точек в **heatmap'ы** (гауссианы).
3. Реализуем **Hourglass**-блок и **Stacked Hourglass Network** с intermediate supervision.
4. Обучаем сеть на MSE и следим за качеством.
5. Декодируем heatmap'ы обратно в координаты точек.
6. Реализуем **выравнивание лица** через similarity-transform (OpenCV) по найденным точкам.
7. Собираем датасет **кропнутых и выровненных лиц** для Задания 2 (с разбиением по личностям).

> ⚙️ **Где запускать.** Архитектура и обучение рассчитаны на GPU (Colab → *Runtime → Change runtime type → GPU*). Датасет CelebA удобно один раз залить на Google Drive и монтировать.

## 0. Окружение и конфигурация

Все «крутилки» проекта собраны в один объект `CFG`, чтобы их было легко менять и чтобы
остальные ноутбуки (Задание 2, доп. задания) использовали **те же** размеры и пути.

In [ ]:
# !pip -q install opencv-python-headless matplotlib pandas tqdm

import os, math, random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# --- Воспроизводимость ---
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

# В Colab монтируем Google Drive (раскомментируйте):
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
@dataclass
class CFG:
    # --- пути (поправьте под себя) ---
    raw_images_dir: str = "/content/drive/MyDrive/celeba/img_celeba"      # сырые картинки CelebA wild
    anno_dir:       str = "/content/drive/MyDrive/celeba/Anno"            # файлы разметки CelebA
    work_dir:       str = "/content/drive/MyDrive/face_project"           # сюда сохраняем результаты

    # --- параметры отбора датасета ---
    min_imgs_per_id: int = 20      # личности с >= стольким числом фото (чтобы хватило на train/val и пары)
    max_imgs_per_id: int = 30      # верхняя отсечка, чтобы датасет не перекосило в сторону «частых» людей
    n_train_ids:    int = 500      # личности для классификации (Задание 2)
    n_query_ids:    int = 40       # личности в query  (доп. задание ID-Rate, НЕ участвуют в обучении)
    n_distractor_ids: int = 400    # личности в distractors (доп. задание ID-Rate)
    bbox_margin:    float = 0.35   # на сколько расширяем bbox при кропе (доля от стороны)

    # --- размеры для Hourglass ---
    input_size:    int = 256       # вход сети
    heatmap_size:  int = 64        # выход сети (1/4 входа — стандарт для hourglass)
    num_landmarks: int = 5         # CelebA: 2 глаза, нос, 2 уголка рта
    sigma:         float = 2.0     # ширина гауссианы в heatmap'е

    # --- размер выровненного лица для Задания 2 ---
    aligned_size:  int = 112

CFG = CFG()
os.makedirs(CFG.work_dir, exist_ok=True)
print("Рабочая папка:", CFG.work_dir)

## 1. Выбор и подготовка датасета

Работаем с **сырой** версией [CelebA (In‑the‑Wild)](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html):
это полноразмерные фотографии (а не уже кропнутые `align&cropped`), к которым приложены
файлы разметки. Нам понадобятся четыре файла из папки `Anno`:

| Файл | Что внутри |
|---|---|
| `identity_CelebA.txt` | `имя_файла  id_личности` — кто изображён (≈10 177 личностей) |
| `list_bbox_celeba.txt` | bounding box лица: `x_1 y_1 width height` |
| `list_landmarks_celeba.txt` | 5 ключевых точек **в координатах исходного изображения** |
| `list_attr_celeba.txt` | 40 бинарных атрибутов (`-1/1`), напр. `Blurry`, `Eyeglasses` |

**Почему важна личность (`identity`).** Для распознавания лиц датасет должен быть размечен
не только «где лицо», но и «кто это». Поэтому в основу отбора кладём именно
`identity_CelebA.txt`: нам нужны люди, у которых **несколько** фотографий.

In [ ]:
def read_celeba_table(path, names=None):
    '''Файлы CelebA: 1-я строка — число записей, 2-я — заголовок (кроме identity).'''
    return pd.read_csv(path, sep=r"\s+", header=None, names=names)

# identity: без заголовка, две колонки
identity = read_celeba_table(os.path.join(CFG.anno_dir, "identity_CelebA.txt"),
                             names=["image_id", "id"])

# bbox / landmarks / attr: пропускаем строку с числом, заголовок берём из 2-й строки
bbox      = pd.read_csv(os.path.join(CFG.anno_dir, "list_bbox_celeba.txt"),
                        sep=r"\s+", skiprows=1)
landmarks = pd.read_csv(os.path.join(CFG.anno_dir, "list_landmarks_celeba.txt"),
                        sep=r"\s+", skiprows=1)
attr      = pd.read_csv(os.path.join(CFG.anno_dir, "list_attr_celeba.txt"),
                        sep=r"\s+", skiprows=1)

# приводим индекс/колонку с именем файла к единому виду 'image_id'
for df in (bbox, landmarks, attr):
    df.reset_index(inplace=True)
    df.rename(columns={"index": "image_id"}, inplace=True)

print("identity :", identity.shape)
print("bbox     :", bbox.shape)
print("landmarks:", landmarks.shape)
print("attr     :", attr.shape)
attr.columns[:6].tolist()

### 1.1. Критерии отбора (обоснование)

Отбор данных — самая важная часть проекта, поэтому фиксируем критерии явно и объясняем их:

1. **Резкость — `Blurry == -1`.** Размытые лица вредят и обучению поиска точек (точка «плывёт»),
   и распознаванию (эмбеддинг шумный). Атрибут `Blurry` — самый дешёвый и надёжный фильтр качества.
2. **Достаточно фото на личность — `≥ min_imgs_per_id` (20).** Для Задания 2 нужно разбить фото
   каждого человека на train/val, а для доп. задания (ID‑Rate, Triplet) — формировать
   позитивные пары/тройки. Поэтому берём только «богатых» на фото людей.
3. **Верхняя отсечка `≤ max_imgs_per_id` (30).** Ограничиваем число фото на человека, чтобы
   классы были сбалансированы и сеть не выучивала пару «звёзд» с сотнями кадров.
4. **Жёсткое разделение личностей на группы (на уровне id, а не картинок):**
   - `train` — личности для обучения распознавания;
   - `query` и `distractors` — личности, **которых модель никогда не видела** (нужно для честной метрики ID‑Rate в доп. задании 1).

   Это разделение мы фиксируем уже здесь, чтобы во всех ноутбуках использовать один и тот же сплит.

Остальные атрибуты (очки, головные уборы, поворот) **намеренно не фильтруем**: устойчивость к ним —
это как раз то, чему сеть должна научиться. Главное — убрать брак (размытость).

In [ ]:
LM_COLS = ["lefteye_x","lefteye_y","righteye_x","righteye_y","nose_x","nose_y",
           "leftmouth_x","leftmouth_y","rightmouth_x","rightmouth_y"]

# 1) объединяем всё в одну таблицу
df = identity.merge(bbox, on="image_id").merge(landmarks, on="image_id").merge(
        attr[["image_id", "Blurry"]], on="image_id")

# 2) убираем размытые
df = df[df["Blurry"] == -1].copy()

# 3) оставляем личности с достаточным числом фото и режем верхушку
counts = df["id"].value_counts()
good_ids = counts[counts >= CFG.min_imgs_per_id].index
df = df[df["id"].isin(good_ids)].copy()
df = df.groupby("id", group_keys=False).head(CFG.max_imgs_per_id)   # не более max на человека

# 4) делим личности на train / query / distractors (детерминированно)
all_ids = sorted(df["id"].unique())
rng = np.random.RandomState(SEED); rng.shuffle(all_ids)
train_ids      = set(all_ids[:CFG.n_train_ids])
query_ids      = set(all_ids[CFG.n_train_ids:CFG.n_train_ids + CFG.n_query_ids])
distractor_ids = set(all_ids[CFG.n_train_ids + CFG.n_query_ids:
                             CFG.n_train_ids + CFG.n_query_ids + CFG.n_distractor_ids])

def group_of(i):
    if i in train_ids: return "train"
    if i in query_ids: return "query"
    if i in distractor_ids: return "distractors"
    return "unused"

df["group"] = df["id"].map(group_of)
df = df[df["group"] != "unused"].copy()

print("Итоговый датасет:", len(df), "изображений,", df['id'].nunique(), "личностей")
print(df["group"].value_counts())
assert len(df) >= 10000, "Нужно 10 000+ картинок — ослабьте критерии в CFG"

### 1.2. Кроп лиц по bbox и сохранение CSV

Сырые фото большие и с фоном. Кропаем лицо по bounding box, **расширяя его на `bbox_margin`**:
небольшое поле вокруг лица важно, потому что (а) разметка bbox не идеальна и точки рта/глаз
иногда у самого края, (б) сети-распознавателю полезен небольшой контекст.

Ключевой технический момент: **ключевые точки в CelebA заданы в координатах исходного
изображения**. После кропа и ресайза их нужно пересчитать в новую систему координат
(вычесть левый-верхний угол кропа и домножить на коэффициент масштаба).

In [ ]:
def crop_face(img, x, y, w, h, margin):
    '''Расширяем bbox на margin (в долях стороны) и кропаем с клиппингом по краям.
       Возвращаем кроп и (ox, oy) — координаты его левого-верхнего угла в исходном фото.'''
    H, W = img.shape[:2]
    dx, dy = int(w * margin), int(h * margin)
    x0, y0 = max(0, x - dx), max(0, y - dy)
    x1, y1 = min(W, x + w + dx), min(H, y + h + dy)
    return img[y0:y1, x0:x1], x0, y0

def landmarks_to_crop(lms_xy, ox, oy, scale_x, scale_y):
    '''Переводим точки из координат исходного фото в координаты ресайзнутого кропа.'''
    out = lms_xy.copy().astype(np.float32)
    out[:, 0] = (out[:, 0] - ox) * scale_x
    out[:, 1] = (out[:, 1] - oy) * scale_y
    return out

crops_dir = os.path.join(CFG.work_dir, "crops")
os.makedirs(crops_dir, exist_ok=True)

records = []
for _, r in df.iterrows():
    img = cv2.imread(os.path.join(CFG.raw_images_dir, r["image_id"]))
    if img is None:
        continue
    crop, ox, oy = crop_face(img, r["x_1"], r["y_1"], r["width"], r["height"], CFG.bbox_margin)
    ch, cw = crop.shape[:2]
    if ch < 10 or cw < 10:
        continue
    crop_resized = cv2.resize(crop, (CFG.input_size, CFG.input_size))
    lms = np.array([[r[LM_COLS[2*k]], r[LM_COLS[2*k+1]]] for k in range(5)], dtype=np.float32)
    lms = landmarks_to_crop(lms, ox, oy, CFG.input_size / cw, CFG.input_size / ch)

    out_name = r["image_id"]
    cv2.imwrite(os.path.join(crops_dir, out_name), crop_resized)
    rec = {"image_id": r["image_id"], "id": r["id"], "group": r["group"]}
    for k in range(5):
        rec[f"x{k}"], rec[f"y{k}"] = float(lms[k, 0]), float(lms[k, 1])
    records.append(rec)

meta = pd.DataFrame(records)
meta.to_csv(os.path.join(CFG.work_dir, "selected_images.csv"), index=False)
print("Сохранено кропов:", len(meta))
print("CSV с исходными именами:", os.path.join(CFG.work_dir, "selected_images.csv"))
meta.head()

> 📦 Файл **`selected_images.csv`** содержит оригинальные имена картинок из CelebA, id личности,
> группу (train/query/distractors) и пересчитанные координаты 5 точек. Именно его требуется сдать
> вместе с проектом, и именно он связывает между собой все ноутбуки.

### 1.3. Визуальная проверка кропов и точек

Всегда полезно глазами убедиться, что точки «легли» правильно после пересчёта координат.

In [ ]:
COLORS = [(255,0,0),(0,128,255),(0,255,0),(255,0,255),(255,255,0)]  # 5 точек
sample = meta.sample(8, random_state=0).reset_index(drop=True)

plt.figure(figsize=(16, 8))
for i, r in sample.iterrows():
    img = cv2.cvtColor(cv2.imread(os.path.join(crops_dir, r["image_id"])), cv2.COLOR_BGR2RGB)
    for k in range(5):
        cv2.circle(img, (int(r[f"x{k}"]), int(r[f"y{k}"])), 3, COLORS[k], -1)
    plt.subplot(2, 4, i + 1); plt.imshow(img); plt.axis("off"); plt.title(f"id {r['id']}")
plt.suptitle("Кропнутые лица и пересчитанные ключевые точки", y=1.02)
plt.tight_layout(); plt.show()

## 2. Из точек — в heatmap'ы

Сеть будет предсказывать не координаты напрямую, а **карты вероятностей (heatmap'ы)** — по одной
на каждую точку. В целевой heatmap'е вокруг истинной точки рисуем гауссиану. Heatmap-подход
устойчивее регрессии к шуму и хорошо обучается под MSE.

Размер heatmap'а — `heatmap_size = 64` (в 4 раза меньше входа `256`), это стандарт для Hourglass.
Значит, координаты точек нужно дополнительно масштабировать в сетку 64×64.

In [ ]:
def create_heatmap(size, landmark, sigma=2):
    '''Один heatmap с гауссовым ядром вокруг точки (size=(H,W), landmark=(x,y)).'''
    x, y = landmark
    h, w = size
    x = min(max(0, int(x)), w - 1); y = min(max(0, int(y)), h - 1)
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    return np.exp(-((yy - y)**2 + (xx - x)**2) / (2 * sigma**2))

def landmarks_to_heatmaps(image_shape, landmarks, sigma=2):
    '''N точек -> массив [N, H, W] из N heatmap'ов.'''
    return np.array([create_heatmap(image_shape, (x, y), sigma) for (x, y) in landmarks])

In [ ]:
# демонстрация: точки кропа (256) -> сетка heatmap'а (64)
r = meta.iloc[0]
scale = CFG.heatmap_size / CFG.input_size
lms64 = [(r[f"x{k}"] * scale, r[f"y{k}"] * scale) for k in range(5)]
hms = landmarks_to_heatmaps((CFG.heatmap_size, CFG.heatmap_size), lms64, sigma=CFG.sigma)

plt.figure(figsize=(16, 3))
names = ["left eye", "right eye", "nose", "left mouth", "right mouth"]
for k in range(5):
    plt.subplot(1, 6, k + 1); plt.imshow(hms[k], cmap="hot"); plt.axis("off"); plt.title(names[k])
plt.subplot(1, 6, 6); plt.imshow(hms.max(0), cmap="hot"); plt.axis("off"); plt.title("все вместе")
plt.suptitle("Целевые heatmap'ы (гауссианы вокруг точек)"); plt.tight_layout(); plt.show()

## 3. `Dataset` и `DataLoader`

`LandmarkDataset` отдаёт пару **(картинка `3×256×256`, целевые heatmap'ы `5×64×64`)**.

Из аугментаций используем безопасные:
* **горизонтальное отражение** — обязательно со «свопом» левых/правых точек (иначе разметка сломается);
* лёгкий **color jitter** (яркость/контраст) — геометрию не трогает.

Геометрические аугментации (поворот/масштаб) тоже возможны, но требуют синхронного пересчёта
точек, поэтому для базовой версии ограничимся флипом — его достаточно для хорошего качества.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
# при флипе индексы меняются местами: (L eye<->R eye), (L mouth<->R mouth), нос на месте
FLIP_IDX = [1, 0, 2, 4, 3]

class LandmarkDataset(Dataset):
    def __init__(self, meta_df, crops_dir, cfg, train=True):
        self.meta = meta_df.reset_index(drop=True)
        self.dir = crops_dir; self.cfg = cfg; self.train = train

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, i):
        r = self.meta.iloc[i]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.dir, r["image_id"])), cv2.COLOR_BGR2RGB)
        lms = np.array([[r[f"x{k}"], r[f"y{k}"]] for k in range(5)], dtype=np.float32)

        if self.train and random.random() < 0.5:                  # горизонтальный флип
            img = img[:, ::-1, :].copy()
            lms[:, 0] = self.cfg.input_size - 1 - lms[:, 0]
            lms = lms[FLIP_IDX]
        if self.train:                                            # лёгкий color jitter
            img = np.clip(img.astype(np.float32) * random.uniform(0.8, 1.2), 0, 255).astype(np.uint8)

        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        x = (x - IMAGENET_MEAN) / IMAGENET_STD

        scale = self.cfg.heatmap_size / self.cfg.input_size
        lms64 = lms * scale
        hms = landmarks_to_heatmaps((self.cfg.heatmap_size, self.cfg.heatmap_size),
                                    lms64, sigma=self.cfg.sigma)
        return x, torch.from_numpy(hms).float()

# для обучения детектора точек берём картинки группы train+query+distractors
# (alignment не зависит от личности, поэтому используем все имеющиеся изображения)
hm_meta = meta.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_val = int(0.1 * len(hm_meta))
val_ds   = LandmarkDataset(hm_meta.iloc[:n_val],  crops_dir, CFG, train=False)
train_ds = LandmarkDataset(hm_meta.iloc[n_val:],  crops_dir, CFG, train=True)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2)
print("train:", len(train_ds), "| val:", len(val_ds))

## 4. Архитектура: Stacked Hourglass Network

**Идея.** Hourglass — это U-Net‑подобная «песочная часовня»: вход сжимается по разрешению до
маленького (захватываем глобальный контекст всего лица), затем разворачивается обратно с
проброс‑связями (skip connections), чтобы вернуть точную локализацию. Каждый «кирпичик» —
`ResidualBlock`.

**Stacked** = несколько Hourglass подряд. После каждого ставится «голова», предсказывающая
heatmap'ы. Это даёт **intermediate supervision**: лосс считается по выходу *каждой* головы, а не
только последней. Градиенты текут со всех голов — сеть учится быстрее и стабильнее.

In [ ]:
class ResidualBlock(nn.Module):
    '''Bottleneck-резидуал: 1x1 -> 3x3 -> 1x1 + skip. Вход и выход одного размера по H,W.'''
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.skip = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)
        self.conv1 = nn.Conv2d(in_channels, out_channels // 2, 1)
        self.bn1   = nn.BatchNorm2d(out_channels // 2)
        self.conv2 = nn.Conv2d(out_channels // 2, out_channels // 2, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_channels // 2)
        self.conv3 = nn.Conv2d(out_channels // 2, out_channels, 1)
        self.bn3   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
    def forward(self, x):
        residual = self.skip(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.bn3(self.conv3(x))
        return self.relu(x + residual)

In [ ]:
class Hourglass(nn.Module):
    '''Рекурсивный hourglass глубины `depth`: симметричный энкодер-декодер из резидуалов
       с проброс-связью на каждом уровне (skip + upsample(low)).'''
    def __init__(self, depth, ch):
        super().__init__()
        self.skip   = ResidualBlock(ch, ch)                      # ветка-проброс на этом разрешении
        self.pool   = nn.MaxPool2d(2, 2)                         # downsample
        self.before = ResidualBlock(ch, ch)
        self.inner  = Hourglass(depth - 1, ch) if depth > 1 else ResidualBlock(ch, ch)
        self.after  = ResidualBlock(ch, ch)
        self.up     = nn.Upsample(scale_factor=2, mode="nearest")  # upsample обратно
    def forward(self, x):
        s = self.skip(x)
        x = self.after(self.inner(self.before(self.pool(x))))
        return s + self.up(x)

In [ ]:
class StackedHourglass(nn.Module):
    def __init__(self, num_stacks=2, num_landmarks=5, ch=256, depth=4):
        super().__init__()
        # стем: 256 -> 64 по разрешению, 3 -> ch по каналам
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3), nn.BatchNorm2d(64), nn.ReLU(True),   # 256 -> 128
            ResidualBlock(64, 128), nn.MaxPool2d(2, 2),                     # 128 -> 64
            ResidualBlock(128, 128), ResidualBlock(128, ch),
        )
        self.hgs   = nn.ModuleList([Hourglass(depth, ch) for _ in range(num_stacks)])
        self.feats = nn.ModuleList([nn.Sequential(ResidualBlock(ch, ch),
                       nn.Conv2d(ch, ch, 1), nn.BatchNorm2d(ch), nn.ReLU(True)) for _ in range(num_stacks)])
        self.heads = nn.ModuleList([nn.Conv2d(ch, num_landmarks, 1) for _ in range(num_stacks)])
        # «слияние» признаков и предсказаний обратно во вход следующего стака
        self.merge_feat = nn.ModuleList([nn.Conv2d(ch, ch, 1) for _ in range(num_stacks - 1)])
        self.merge_pred = nn.ModuleList([nn.Conv2d(num_landmarks, ch, 1) for _ in range(num_stacks - 1)])

    def forward(self, x):
        x = self.stem(x)
        outputs = []
        for i in range(len(self.hgs)):
            feat = self.feats[i](self.hgs[i](x))
            pred = self.heads[i](feat)            # heatmap'ы этой головы
            outputs.append(pred)
            if i < len(self.hgs) - 1:             # intermediate supervision: прокидываем дальше
                x = x + self.merge_feat[i](feat) + self.merge_pred[i](pred)
        return outputs                             # список heatmap'ов по числу стаков

**Проверка форм (sanity-check).** Прогоним случайный батч и убедимся, что сеть возвращает
`num_stacks` голов, каждая — `[B, 5, 64, 64]`. Ниже — реальный вывод этой ячейки.

In [1]:
model = StackedHourglass(num_stacks=2, num_landmarks=CFG.num_landmarks, ch=256, depth=4).to(DEVICE)
x = torch.randn(2, 3, CFG.input_size, CFG.input_size).to(DEVICE)
outs = model(x)
print("input:", tuple(x.shape))
print("num heatmap outputs (stacks):", len(outs))
for i, o in enumerate(outs):
    print(f"  stack {i} heatmaps:", tuple(o.shape))
print("params: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

input: (2, 3, 256, 256)
num heatmap outputs (stacks): 2
  stack 0 heatmaps: (2, 5, 64, 64)
  stack 1 heatmaps: (2, 5, 64, 64)
params: 6.56M


## 5. Обучение

**Лосс.** MSE между предсказанной и целевой heatmap'ой, **просуммированный по всем головам**
(intermediate supervision). PyTorch сам построит граф и пустит градиенты во все стаки:

```
outputs = model(image)            # список heatmap'ов от голов
loss = sum(MSE(out, target) for out in outputs)
loss.backward(); optimizer.step()
```

**Оптимизатор** — Adam, `lr=2.5e-4` (классическое значение для Hourglass), косинусный
прогрев/затухание по эпохам. Не забываем сохранять чекпойнт лучшей модели.

In [ ]:
def hm_loss(outputs, target):
    return sum(F.mse_loss(o, target) for o in outputs)

def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    total, n = 0.0, 0
    torch.set_grad_enabled(train)
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        out = model(x)
        loss = hm_loss(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * x.size(0); n += x.size(0)
    torch.set_grad_enabled(True)
    return total / n

In [ ]:
EPOCHS = 30
opt = torch.optim.Adam(model.parameters(), lr=2.5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
ckpt_path = os.path.join(CFG.work_dir, "hourglass_best.pt")

history = {"train": [], "val": []}
best = float("inf")
for ep in range(EPOCHS):
    tr = run_epoch(model, train_loader, opt)
    va = run_epoch(model, val_loader)
    sched.step()
    history["train"].append(tr); history["val"].append(va)
    if va < best:
        best = va; torch.save(model.state_dict(), ckpt_path)
    print(f"epoch {ep+1:02d}/{EPOCHS}  train {tr:.5f}  val {va:.5f}  (best {best:.5f})")
print("Лучшая модель сохранена:", ckpt_path)

> 💬 **Заметка по экспериментам (заполните своими наблюдениями).** Здесь стоит коротко описать
> ваш реальный путь: с каким `lr` стартовали, помог ли флип, не переобучается ли сеть, сколько
> эпох хватило. Это именно та «логика движения», которую просят показать в проекте.

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(history["train"], label="train")
plt.plot(history["val"], label="val")
plt.xlabel("эпоха"); plt.ylabel("MSE (сумма по головам)")
plt.title("Кривая обучения Stacked Hourglass"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 6. Декодирование heatmap → координаты

Из предсказанной heatmap'ы координату точки берём как позицию максимума (`argmax`). Для
субпиксельной точности можно сместить максимум на полпикселя в сторону соседа с большим
значением (приём из оригинальной статьи) — приведён ниже. Координаты в сетке 64 умножаем
на 4, чтобы вернуться в пространство кропа 256.

In [ ]:
def decode_heatmaps(hm, input_size, heatmap_size):
    '''hm: [K,H,W] (numpy или tensor) -> точки [K,2] в координатах входа (input_size).'''
    if torch.is_tensor(hm):
        hm = hm.detach().cpu().numpy()
    K, H, W = hm.shape
    pts = np.zeros((K, 2), np.float32)
    for k in range(K):
        y, x = np.unravel_index(hm[k].argmax(), (H, W))
        # субпиксельное уточнение по соседям
        if 1 < x < W - 1 and hm[k][y, x+1] != hm[k][y, x-1]:
            x += 0.25 * np.sign(hm[k][y, x+1] - hm[k][y, x-1])
        if 1 < y < H - 1 and hm[k][y+1, x] != hm[k][y-1, x]:
            y += 0.25 * np.sign(hm[k][y+1, x] - hm[k][y-1, x])
        pts[k] = [x, y]
    return pts * (input_size / heatmap_size)

# визуальное сравнение GT (зелёный) и предсказанных (красный) точек на валидации
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)); model.eval()
plt.figure(figsize=(16, 8))
for i in range(8):
    x, y = val_ds[i]
    with torch.no_grad():
        pred = model(x.unsqueeze(0).to(DEVICE))[-1][0]      # последняя голова
    pts = decode_heatmaps(pred, CFG.input_size, CFG.heatmap_size)
    gt  = decode_heatmaps(y,    CFG.input_size, CFG.heatmap_size)
    img = ((x * IMAGENET_STD + IMAGENET_MEAN).permute(1,2,0).numpy() * 255).clip(0,255).astype(np.uint8).copy()
    for k in range(5):
        cv2.circle(img, tuple(gt[k].astype(int)),  4, (0,255,0), -1)
        cv2.circle(img, tuple(pts[k].astype(int)), 2, (255,0,0), -1)
    plt.subplot(2,4,i+1); plt.imshow(img); plt.axis("off")
plt.suptitle("Зелёный — истинные точки, красный — предсказанные", y=1.02)
plt.tight_layout(); plt.show()

## 7. Выравнивание лица (face alignment)

Это **задача классического CV**, без нейросетей. По 5 найденным точкам ищем геометрическое
преобразование, которое переводит их в фиксированный **эталонный шаблон** (canonical template),
и применяем его ко всей картинке (`warp`).

Берём **similarity-transform** (поворот + единый масштаб + сдвиг, без перекоса) — он надёжно
работает по 5 точкам и не искажает пропорции лица. В OpenCV это
`cv2.estimateAffinePartial2D`. Эталонный шаблон — стандартные 5 точек ArcFace для лица `112×112`.

Ниже — sanity-check самого преобразования на синтетике (реальный вывод ячейки): берём шаблон,
искусственно поворачиваем/масштабируем/сдвигаем, и проверяем, что выравнивание возвращает точки
ровно на шаблон (ошибка ≈ 0).

In [2]:
# эталонные 5 точек ArcFace для выровненного лица 112x112
REFERENCE_5PTS = np.array([
    [38.2946, 51.6963],   # левый глаз
    [73.5318, 51.5014],   # правый глаз
    [56.0252, 71.7366],   # нос
    [41.5493, 92.3655],   # левый уголок рта
    [70.7299, 92.2041],   # правый уголок рта
], dtype=np.float32)

def align_face(img, landmarks, out_size, ref=REFERENCE_5PTS):
    '''Similarity-transform по 5 точкам -> warp к шаблону out_size x out_size.'''
    scale = out_size / 112.0
    src = np.asarray(landmarks, dtype=np.float32)
    M, _ = cv2.estimateAffinePartial2D(src, ref * scale, method=cv2.LMEDS)
    aligned = cv2.warpAffine(img, M, (out_size, out_size), borderValue=0)
    return aligned, M

# --- sanity-check на синтетике ---
ang, s, t = np.deg2rad(20.0), 1.7, np.array([60.0, 40.0])
R = s * np.array([[np.cos(ang), -np.sin(ang)], [np.sin(ang), np.cos(ang)]])
detected = REFERENCE_5PTS @ R.T + t
_, M = align_face(np.zeros((300,300,3), np.uint8), detected, out_size=112)
mapped = (M @ np.hstack([detected, np.ones((5,1))]).T).T
err = np.linalg.norm(mapped - REFERENCE_5PTS, axis=1)
print("per-point reprojection error (px):", np.round(err, 4))
print("max error: %.5f px" % err.max())

per-point reprojection error (px): [0. 0. 0. 0. 0.]
max error: 0.00000 px


### 7.1. Примеры работы выравнивания «было → стало»

Прогоняем кропы через обученную сеть, декодируем точки, выравниваем. Видно, что у всех лиц
глаза встают на одну горизонталь, а масштаб становится одинаковым.

In [ ]:
plt.figure(figsize=(12, 9))
for i in range(6):
    x, _ = val_ds[i]
    with torch.no_grad():
        pred = model(x.unsqueeze(0).to(DEVICE))[-1][0]
    pts = decode_heatmaps(pred, CFG.input_size, CFG.heatmap_size)
    img = ((x*IMAGENET_STD+IMAGENET_MEAN).permute(1,2,0).numpy()*255).clip(0,255).astype(np.uint8).copy()
    aligned, _ = align_face(img, pts, out_size=CFG.aligned_size)
    plt.subplot(3, 4, 2*i+1); plt.imshow(img);      plt.axis("off"); plt.title("было")
    plt.subplot(3, 4, 2*i+2); plt.imshow(aligned);  plt.axis("off"); plt.title("стало")
plt.suptitle("Выравнивание лиц по предсказанным точкам", y=1.01); plt.tight_layout(); plt.show()

## 8. Сборка датасета выровненных лиц для Задания 2

Финальный шаг — прогнать **все** отобранные лица через `детекция-кроп(уже сделан) → точки → выравнивание`
и разложить результат по папкам так, чтобы:

* `aligned/train/<id>/...` и `aligned/val/<id>/...` — личности группы **train** (фото каждого
  человека делим 85/15 на train/val для классификации в Задании 2);
* `aligned/query/<id>/...` и `aligned/distractors/<id>/...` — **невидимые** при обучении личности
  для метрики ID‑Rate (доп. задание 1).

Разделение происходит **на уровне папок** — так мы гарантированно знаем, на чём модель училась,
а на чём нет.

In [ ]:
from collections import defaultdict
import shutil

aligned_root = os.path.join(CFG.work_dir, "aligned")
if os.path.exists(aligned_root):
    shutil.rmtree(aligned_root)

# для train-личностей заранее решаем, какие фото пойдут в val (15%)
val_pick = set()
for pid, g in meta[meta.group == "train"].groupby("id"):
    names = g["image_id"].tolist(); random.Random(pid).shuffle(names)
    val_pick.update(names[:max(1, int(0.15 * len(names)))])

def split_dir(row):
    if row["group"] == "train":
        return "val" if row["image_id"] in val_pick else "train"
    return row["group"]                       # query / distractors

count = defaultdict(int)
for _, r in meta.iterrows():
    img = cv2.imread(os.path.join(crops_dir, r["image_id"]))
    if img is None:
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        x = ((torch.from_numpy(img).permute(2,0,1).float()/255.0 - IMAGENET_MEAN)/IMAGENET_STD)
        pred = model(x.unsqueeze(0).to(DEVICE))[-1][0]
    pts = decode_heatmaps(pred, CFG.input_size, CFG.heatmap_size)
    aligned, _ = align_face(img, pts, out_size=CFG.aligned_size)

    sub = split_dir(r)
    d = os.path.join(aligned_root, sub, str(r["id"]))
    os.makedirs(d, exist_ok=True)
    cv2.imwrite(os.path.join(d, r["image_id"]), cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR))
    count[sub] += 1

print("Готово. Выровненных лиц по папкам:", dict(count))
print("Папка с результатом:", aligned_root)

## Итоги

* Отобрали и обосновали датасет **10 000+** лиц CelebA, сохранили `selected_images.csv` с исходными именами.
* Реализовали `ResidualBlock → Hourglass → StackedHourglass` с intermediate supervision (проверили формы).
* Обучили детектор 5 точек на MSE по heatmap'ам, показали предсказания vs истину.
* Сделали выравнивание similarity-transform’ом по эталонному шаблону (sanity-check ошибки ≈ 0) и привели примеры «было → стало».
* Собрали датасет выровненных лиц `aligned/{train,val,query,distractors}/<id>/...` — он используется во всех следующих ноутбуках.

**Артефакты:** `hourglass_best.pt`, `selected_images.csv`, папка `aligned/`.

➡️ Дальше — **Задание 2 (`2_ArcFace.ipynb`)**: обучаем сеть-распознаватель на CE и ArcFace.